In [1]:
# =============================================================================
# FULL PIPELINE: Raw Configuration Extraction -> Calibration Standardization -> Mapping
# =============================================================================
# This notebook demonstrates the complete workflow:
# 1. Read raw file configurations and save them
# 2. Read manufacturer calibration files and save to single-channel standardized files
# 3. Load single-channel files and match raw file channels to calibration data
# 4. Show how to use the resulting mapping files
#
# The single-channel calibration files produced in step 2 are the canonical
# intermediate format.  The same files can also be provided directly by a user
# (see user_provided_cal_pipeline.ipynb) without running steps 1-2.
# =============================================================================

import sys
from pathlib import Path

# Add the package source directory to path so aa_si_calibration is importable
# (not needed if the package is installed via pip install -e .)
sys.path.insert(0, str(Path("../src").resolve()))

# Raw file reading
from aa_si_calibration.raw_reader_api import process_raw_folder, save_yaml

# Calibration file parsing and standardized format
from aa_si_calibration import manufacturer_file_parsers
from aa_si_calibration import standardized_file_lib

# Mapping algorithm and pipeline functions
from aa_si_calibration.mapping_algorithm import (
    load_raw_configs,
    load_calibration_data_from_single_files,
    build_mapping,
    get_calibration,
    get_calibration_from_file,
    save_mapping_files,
    print_mapping_preview,
    handle_unused_calibration_files,
    resolve_conflicts_interactive,
    check_for_conflicts,
    check_required_calibration_params,
    verify_calibration_file_usage,
)

from aa_si_calibration.standardized_file_lib import (
    remap_to_short_keys,
    print_short_key_summary,
)

print("All imports successful!")

All imports successful!


In [2]:
# =============================================================================
# CONFIGURATION - Define Input and Output Paths
# =============================================================================

# If True, unused/rejected calibration files are moved to an "unused" folder
# instead of being permanently deleted.
KEEP_UNUSED_STANDARDIZED_FILES = True

# Record author - the individual running this pipeline and generating the records
RECORD_AUTHOR = "Placeholder Author"

# Cruise identifier for the calibration records
CRUISE_ID = "Pipeline_Example"

# Filename style for single-channel calibration files:
#   False -> long filenames derived from the full calibration key
#   True  -> short filenames: <date>_<freq_hz>_config-<N>.yml
SHORT_FILENAMES = True

# Conflict resolution strategy when a raw channel matches multiple cal files:
#   "interactive" -> prompt the user to choose which file to keep
#   "error"       -> raise an error listing the conflicts; the user must
#                     manually delete the unwanted file(s) and re-run the cell
CONFLICT_RESOLUTION = "error"

# Input folders
RAW_INPUT_FOLDER = Path("./example_data/ek80_FM_raw_file_input_folder")
# RAW_INPUT_FOLDER = Path("./example_data/ek80_CW_raw_file_input_folder")

CAL_INPUT_FOLDER = Path("./example_data/ek80_FM_cal_file_input_folder")
# CAL_INPUT_FOLDER = Path("./example_data/ek80_cal_file_input_folder")

# Output folder with subfolders for organization
OUTPUT_BASE = Path("./Full_Pipeline_Outputs")

# Create output subdirectories
RAW_CONFIGS_OUTPUT = OUTPUT_BASE / "raw_file_configs"
SINGLE_CAL_OUTPUT = OUTPUT_BASE / "single_channel_calibration_files"
MAPPING_OUTPUT = OUTPUT_BASE / "mapping_files"
UNUSED_CAL_OUTPUT = OUTPUT_BASE / "unused_calibration_files"
LOGS_OUTPUT = OUTPUT_BASE / "logs"

# Create all output directories
for folder in [RAW_CONFIGS_OUTPUT, SINGLE_CAL_OUTPUT, MAPPING_OUTPUT, LOGS_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

# Create unused folder only when keeping files
if KEEP_UNUSED_STANDARDIZED_FILES:
    UNUSED_CAL_OUTPUT.mkdir(parents=True, exist_ok=True)

# Output file paths
RAW_CONFIGS_PATH = RAW_CONFIGS_OUTPUT / "raw_file_configs.yaml"
MAPPING_PATH = MAPPING_OUTPUT / "channel_to_calibration_mapping.yaml"
CALIBRATION_DICT_PATH = MAPPING_OUTPUT / "calibration_configurations.yaml"

# Global calibration parameters applied to all single-channel files
global_params = {
    "cruise_id": CRUISE_ID,
    "record_author": RECORD_AUTHOR,
}

print(f"Record author: {RECORD_AUTHOR}")
print(f"Cruise ID: {CRUISE_ID}")
print(f"Short filenames: {SHORT_FILENAMES}")
print(f"Keep unused standardized files: {KEEP_UNUSED_STANDARDIZED_FILES}")
print(f"Conflict resolution mode: {CONFLICT_RESOLUTION}")
print(f"\nInput folders:")
print(f"  Raw files:        {RAW_INPUT_FOLDER}")
print(f"  Calibration files: {CAL_INPUT_FOLDER}")
print(f"\nOutput folders:")
print(f"  Raw configs:              {RAW_CONFIGS_OUTPUT}")
print(f"  Single-channel cal files: {SINGLE_CAL_OUTPUT}")
print(f"  Mapping files:            {MAPPING_OUTPUT}")
print(f"  Logs:                     {LOGS_OUTPUT}")
if KEEP_UNUSED_STANDARDIZED_FILES:
    print(f"  Unused cal files:         {UNUSED_CAL_OUTPUT}")

Record author: Placeholder Author
Cruise ID: Pipeline_Example
Short filenames: True
Keep unused standardized files: True
Conflict resolution mode: error

Input folders:
  Raw files:        example_data\ek80_FM_raw_file_input_folder
  Calibration files: example_data\ek80_FM_cal_file_input_folder

Output folders:
  Raw configs:              Full_Pipeline_Outputs\raw_file_configs
  Single-channel cal files: Full_Pipeline_Outputs\single_channel_calibration_files
  Mapping files:            Full_Pipeline_Outputs\mapping_files
  Logs:                     Full_Pipeline_Outputs\logs
  Unused cal files:         Full_Pipeline_Outputs\unused_calibration_files


In [3]:
# =============================================================================
# STEP 1: READ RAW FILE CONFIGURATIONS
# =============================================================================
# Process all .raw files in the input folder and extract channel configurations.
# Results are sorted by metadata_start_time (earliest first).

file_configs, frequencies_set = process_raw_folder(RAW_INPUT_FOLDER, verbose=True)

# Save raw file configurations to YAML
save_yaml(file_configs, RAW_CONFIGS_PATH)
print(f"\n Saved raw file configurations to: {RAW_CONFIGS_PATH}")

Found 2 raw files in example_data\ek80_FM_raw_file_input_folder
  - D20230721-T174615.raw
  - D20230721-T174634.raw
File: D20230721-T174615.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 313
  Valid GPS fixes: 20
  First GPS: 41.515678, -71.340823
File: D20230721-T174634.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 314
  Valid GPS fixes: 20
  First GPS: 41.515038, -71.341385

SUMMARY: Processed 2 files (sorted by metadata_start_time)
Unique frequencies found: [18000.0, 38000.0, 70000.0, 200000.0] Hz

 Saved raw file configurations to: Full_Pipeline_Outputs\raw_file_configs\raw_file_configs.yaml


In [4]:
# =============================================================================
# STEP 2: READ MANUFACTURER CALIBRATION FILES AND SAVE AS SINGLE-CHANNEL FILES
# =============================================================================
# Automatically detects EK60 (.cal) or EK80 (.xml) calibration files,
# parses them, validates against the standardized schema, and saves each
# channel as an individual single-channel .yml file.
#
# These single-channel files are the canonical intermediate format shared
# by both this pipeline (manufacturer files) and the user-provided pipeline.

# Auto-detect calibration file type and extract parameters
# (frequencies from raw files are passed for EK60 sorting; the library sorts them)
cal_params, env_params, other_params, cal_file_type = \
    manufacturer_file_parsers.extract_and_convert_calibration_params(
        CAL_INPUT_FOLDER,
        nc_frequencies=frequencies_set,
        output_logs_folder=LOGS_OUTPUT
    )

print("\n" + "=" * 80)
print(f"Parsed {cal_file_type} calibration parameters summary:")
print("=" * 80)
print(f"Channels: {other_params.get('channel')}")
print(f"Frequencies: {other_params.get('frequency_nominal')}")
print(f"Gain corrections: {cal_params.get('gain_correction')}")
print(f"Sa corrections: {cal_params.get('sa_correction')}")
print(f"Equivalent beam angles: {cal_params.get('equivalent_beam_angle')}")

Detected calibration file type: EK80
Found 9 EK80 XML calibration files in example_data\ek80_FM_cal_file_input_folder

Parsing: CalibrationDataFile-D20230720-T183638_cw_onaxis200_only.xml
   Extracted parameters for ES200-7C_200-200kHz
      Frequency range: 200000 - 200000 Hz (1 points)
      Gain range: 26.86 - 26.86 dB
      Power: 150 W
      Pulse Form: CW

Parsing: CalibrationDataFile-D20230720-T184613_200FM_1ms_full.xml
   Extracted parameters for ES200-7C_160-260kHz
      Frequency range: 160000 - 260000 Hz (65 points)
      Gain range: 22.07 - 29.01 dB
      Power: 150 W
      Pulse Form: LFM

Parsing: CalibrationDataFile-D20230720-T192325_70cw_onaxis.xml
   Extracted parameters for ES70-7C_70-70kHz
      Power: 750 W
      Pulse Form: CW

Parsing: CalibrationDataFile-D20230720-T193008-70khzFM_try1_got_snagged.xml
   Extracted parameters for ES70-7C_45-90kHz
      Frequency range: 45000 - 85630 Hz (83 points)
      Gain range: 22.79 - 29.73 dB
      Power: 750 W
      Pulse Fo

In [5]:
# =============================================================================
# STEP 2 (continued): Save calibration data as single-channel files
# =============================================================================
# Each channel is saved as an individual .yml file. These files serve as the
# canonical intermediate format — the same format a user can provide directly
# in the user-provided calibration pipeline.

saved_count, _, standardized_dict = standardized_file_lib.save_single_channel_files_from_params(
    cal_params, 
    env_params, 
    other_params, 
    global_params, 
    output_dir=SINGLE_CAL_OUTPUT,
    short_filenames=SHORT_FILENAMES,
)

print(f"\n Saved {saved_count} single-channel calibration file(s) to: {SINGLE_CAL_OUTPUT}")

# List the saved files
print("\n" + "=" * 80)
print("Single-channel calibration files:")
print("=" * 80)
for f in sorted(SINGLE_CAL_OUTPUT.glob("*.yml")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.1f} KB)")

global_param cruise_id
global_param record_author ignored because it is reserved
data validated by json schema

⚠️  WARNING: 1 calibration key(s) appeared more than once (same configuration from multiple source files).
   Disambiguation suffixes (__1, __2, …) have been appended.

   Key: 2023-07-20__ES70-7C Serial No: 0__NoSN__1__0.001024__750.0__45000.0__90000.0
      __1: ['CalibrationDataFile-D20230720-T193008-70khzFM_try1_got_snagged.xml']
      __2: ['CalibrationDataFile-D20230720-T221812-70FM_try2.xml']

 Saved 9 single-channel calibration file(s) to: Full_Pipeline_Outputs\single_channel_calibration_files

Single-channel calibration files:
  2023-07-20__18000__config-1.yml (1.6 KB)
  2023-07-20__200000__config-1.yml (1.7 KB)
  2023-07-20__200000__config-2.yml (5.4 KB)
  2023-07-20__38000__config-1.yml (1.7 KB)
  2023-07-20__38000__config-2.yml (8.1 KB)
  2023-07-20__70000__config-1.yml (1.6 KB)
  2023-07-20__70000__config-2__1.yml (6.4 KB)
  2023-07-20__70000__config-2__2.yml (6.

In [6]:
# =============================================================================
# STEP 3: LOAD RAW FILE CONFIGURATIONS
# =============================================================================
# Load the raw file configurations we saved in Step 1.
# Calibration data is loaded in the next cell (together with the mapping step)
# so that deleting calibration files and re-running that cell is sufficient.

raw_file_configs = load_raw_configs(RAW_CONFIGS_PATH)

print(f"Loaded {len(raw_file_configs)} raw file configurations")
print(f"\nRaw files: {[f['filename'] for f in raw_file_configs]}")

Loaded 2 raw file configurations

Raw files: ['D20230721-T174615.raw', 'D20230721-T174634.raw']


In [7]:
# =============================================================================
# STEP 3 (continued): LOAD CALIBRATION DATA AND BUILD MAPPING
# =============================================================================
# Loads the single-channel calibration files, then matches raw channels to
# calibration data.  If any raw channel matches MULTIPLE calibration files,
# the conflict is resolved according to CONFLICT_RESOLUTION:
#   "interactive" -> you will be prompted to choose which file to keep
#   "error"       -> a ValueError is raised listing the conflicts; delete
#                     the unwanted file(s) from SINGLE_CAL_OUTPUT and re-run
#                     this cell

# Load calibration data from the single-channel files directory
calibration_data = load_calibration_data_from_single_files(SINGLE_CAL_OUTPUT)

print(f"Loaded {len(calibration_data['channels'])} calibration channel(s) "
      f"from {SINGLE_CAL_OUTPUT}")

# Build the mapping
result = build_mapping(raw_file_configs, calibration_data, verbose=False)
result.print_summary()

# Handle unused calibration files (move or delete)
handle_unused_calibration_files(
    result, calibration_data, SINGLE_CAL_OUTPUT,
    keep_unused=KEEP_UNUSED_STANDARDIZED_FILES,
    unused_dir=UNUSED_CAL_OUTPUT,
)

# Resolve any multiple-match conflicts
if CONFLICT_RESOLUTION == "interactive":
    resolve_conflicts_interactive(
        result, SINGLE_CAL_OUTPUT,
        keep_unused=KEEP_UNUSED_STANDARDIZED_FILES,
        unused_dir=UNUSED_CAL_OUTPUT,
    )
elif CONFLICT_RESOLUTION == "error":
    check_for_conflicts(result, cal_files_dir=SINGLE_CAL_OUTPUT)
else:
    raise ValueError(
        f"Unknown CONFLICT_RESOLUTION mode: {CONFLICT_RESOLUTION!r}. "
        f"Use 'interactive' or 'error'."
    )

# Access the dictionaries from the result
mapping_dict = result.mapping_dict
calibration_dict = result.calibration_dict

Loaded 9 calibration channel(s) from Full_Pipeline_Outputs\single_channel_calibration_files

MATCHING SUMMARY

Raw file channels:
  Total file channels processed: 8
  Total unique channels: 4
  Matched file channels: 8
  Matched unique channels: 4
  Unmatched file channels: 0
  Multiplexing warnings: 0

Calibration files:
  Total calibrations loaded: 9
  Unique calibrations used: 6
  Conflicting calibration sets: 2

3 manufacturer calibration file(s) not matched by any raw channel (moved to unused folder):
  - CalibrationDataFile-D20230720-T183638_cw_onaxis200_only.xml
  - CalibrationDataFile-D20230720-T192325_70cw_onaxis.xml
  - CalibrationDataFile-D20230720-T232023-38CW_onaxis.xml

Moved 3 unused file(s) to: Full_Pipeline_Outputs\unused_calibration_files

CONFLICT: MULTIPLE CALIBRATION MATCHES DETECTED

2 unique raw configuration(s) matched multiple calibration files.
Each raw configuration must match exactly ONE calibration file.
Delete the unwanted file(s) from:
  Full_Pipeline_Out

ValueError: Found 2 unique raw configuration(s) with multiple calibration matches in Full_Pipeline_Outputs\single_channel_calibration_files. Delete the extras and re-run this step.

In [ ]:
# =============================================================================
# STEP 3 (continued): Save mapping files and preview results
# =============================================================================

# Preview the mapping results
print_mapping_preview(result)

# Save the mapping and calibration dictionaries
mapping_path, calibration_path = save_mapping_files(
    result, MAPPING_OUTPUT, short_filenames=SHORT_FILENAMES
)

print(f"\nSaved mapping dictionary to: {mapping_path}")
print(f"Saved calibration dictionary to: {calibration_path}")
print(f"\nNote: Single-channel calibration files already exist in: {SINGLE_CAL_OUTPUT}")

# When using short filenames, update in-memory dicts to match the saved files
if SHORT_FILENAMES:
    mapping_dict, calibration_dict, short_map = remap_to_short_keys(
        mapping_dict, calibration_dict
    )
    print_short_key_summary(short_map, result.calibration_dict)

MAPPING DICTIONARY PREVIEW

D20230721-T174615.raw:
  WBT 400479-15 ES18_ES
    -> 2023-07-20__18000__config-1
  WBT 400528-15 ES38-7_ES
    -> 2023-07-20__38000__config-2
  WBT 400503-15 ES70-7C_ES
    -> 2023-07-20__70000__config-2__2
  WBT 400509-15 ES200-7C_ES
    -> 2023-07-20__200000__config-2

D20230721-T174634.raw:
  WBT 400479-15 ES18_ES
    -> 2023-07-20__18000__config-1
  WBT 400528-15 ES38-7_ES
    -> 2023-07-20__38000__config-2
  WBT 400503-15 ES70-7C_ES
    -> 2023-07-20__70000__config-2__2
  WBT 400509-15 ES200-7C_ES
    -> 2023-07-20__200000__config-2


CALIBRATION DICTIONARY KEYS
  2023-07-20__18000__config-1
  2023-07-20__38000__config-2
  2023-07-20__70000__config-2__2
  2023-07-20__200000__config-2

Saved mapping dictionary to: Full_Pipeline_Outputs\mapping_files\channel_to_calibration_mapping.yaml
Saved calibration dictionary to: Full_Pipeline_Outputs\mapping_files\calibration_configurations.yaml

Note: Single-channel calibration files already exist in: Full_Pipelin

In [ ]:
# =============================================================================
# CHECK FOR MISSING REQUIRED CALIBRATION PARAMETERS
# =============================================================================
# For each calibration entry used in the mapping, check that all required
# calibration parameters are present (non-null).  Mapping/configuration
# parameters (transceiver_id, transducer_model, etc.) are excluded — they
# were already validated during the matching step.

missing = check_required_calibration_params(calibration_dict)

MISSING REQUIRED CALIBRATION PARAMETER CHECK

 All required calibration parameters are present for every mapped channel.


In [ ]:
# =============================================================================
# VERIFY: ALL REMAINING CALIBRATION FILES ARE USED
# =============================================================================
# Confirm that every single-channel calibration file in SINGLE_CAL_OUTPUT is
# referenced by the mapping.

unused_files = verify_calibration_file_usage(calibration_dict, SINGLE_CAL_OUTPUT)

CALIBRATION FILE USAGE CHECK

  3 single-channel calibration file(s) are NOT used in the mapping:
     - 2023-07-20__200000__config-2.yml
     - 2023-07-20__38000__config-2.yml
     - 2023-07-20__70000__config-2__2.yml

   Re-run the mapping step to resolve these.


In [ ]:
# =============================================================================
# STEP 4: DEMONSTRATE HOW TO USE THE MAPPING FILES
# =============================================================================
# This section shows how a user can load and use the mapping files
# to retrieve calibration data for any raw file + channel combination

print("=" * 80)
print("HOW TO USE THE MAPPING FILES")
print("=" * 80)
print("""
The pipeline produces several output files that can be used independently:

1. RAW FILE CONFIGURATIONS (raw_file_configs.yaml):
   - Contains channel configurations extracted from each raw file
   - Useful for understanding what channels/frequencies are in your data

2. SINGLE-CHANNEL CALIBRATION FILES (single_channel_calibration_files/):
   - One .yml file per channel with calibration parameters
   - This is the canonical intermediate format
   - Can be provided directly by a user (see user_provided_cal_pipeline.ipynb)

3. MAPPING DICTIONARY (channel_to_calibration_mapping.yaml):
   - Maps each raw file + channel to a calibration configuration key
   - Allows quick lookup of which calibration to use

4. CALIBRATION CONFIGURATIONS (calibration_configurations.yaml):
   - Contains the actual calibration parameters indexed by key
   - Used together with the mapping dictionary
""")

print("\n" + "=" * 80)
print("EXAMPLE: Loading and using the mapping files")
print("=" * 80)

HOW TO USE THE MAPPING FILES

The pipeline produces several output files that can be used independently:

1. RAW FILE CONFIGURATIONS (raw_file_configs.yaml):
   - Contains channel configurations extracted from each raw file
   - Useful for understanding what channels/frequencies are in your data

2. SINGLE-CHANNEL CALIBRATION FILES (single_channel_calibration_files/):
   - One .yml file per channel with calibration parameters
   - This is the canonical intermediate format
   - Can be provided directly by a user (see user_provided_cal_pipeline.ipynb)

3. MAPPING DICTIONARY (channel_to_calibration_mapping.yaml):
   - Maps each raw file + channel to a calibration configuration key
   - Allows quick lookup of which calibration to use

4. CALIBRATION CONFIGURATIONS (calibration_configurations.yaml):
   - Contains the actual calibration parameters indexed by key
   - Used together with the mapping dictionary


EXAMPLE: Loading and using the mapping files


In [ ]:
# =============================================================================
# EXAMPLE USAGE: Retrieve calibration for a specific raw file + channel
# =============================================================================

# First, let's see what raw files and channels are available
print("Available raw files and channels:")
for filename, channels in mapping_dict.items():
    print(f"\n  {filename}:")
    for channel_id, cal_key in channels.items():
        print(f"    - {channel_id}")
        print(f"      -> maps to: {cal_key}")

# Now demonstrate retrieving calibration for a specific file/channel
print("\n" + "=" * 80)
print("RETRIEVING CALIBRATION DATA")
print("=" * 80)

# Get the first file and first channel as an example
if mapping_dict:
    example_filename = list(mapping_dict.keys())[0]
    if mapping_dict[example_filename]:
        example_channel = list(mapping_dict[example_filename].keys())[0]
        
        # Method 1: Use the in-memory dictionaries
        cal_data = get_calibration(example_filename, example_channel, mapping_dict, calibration_dict)
        
        if cal_data:
            print(f"\nCalibration for '{example_filename}' -> '{example_channel}':")
            print(f"  Calibration date: {cal_data.get('calibration_date')}")
            print(f"  Frequency: {cal_data.get('frequency')}")
            print(f"  Gain correction: {cal_data.get('gain_correction')}")
            print(f"  Sa correction: {cal_data.get('sa_correction')}")
            print(f"  Equivalent beam angle: {cal_data.get('equivalent_beam_angle')}")
            print(f"  Sound speed: {cal_data.get('sound_speed_indicative')}")
            print(f"  Absorption: {cal_data.get('absorption_indicative')}")
        else:
            print(f"No calibration found for {example_filename} -> {example_channel}")

Available raw files and channels:

  D20230721-T174615.raw:
    - WBT 400479-15 ES18_ES
      -> maps to: 2023-07-20__18000__config-1
    - WBT 400528-15 ES38-7_ES
      -> maps to: 2023-07-20__38000__config-1
    - WBT 400503-15 ES70-7C_ES
      -> maps to: 2023-07-20__70000__config-1
    - WBT 400509-15 ES200-7C_ES
      -> maps to: 2023-07-20__200000__config-1

  D20230721-T174634.raw:
    - WBT 400479-15 ES18_ES
      -> maps to: 2023-07-20__18000__config-1
    - WBT 400528-15 ES38-7_ES
      -> maps to: 2023-07-20__38000__config-1
    - WBT 400503-15 ES70-7C_ES
      -> maps to: 2023-07-20__70000__config-1
    - WBT 400509-15 ES200-7C_ES
      -> maps to: 2023-07-20__200000__config-1

RETRIEVING CALIBRATION DATA

Calibration for 'D20230721-T174615.raw' -> 'WBT 400479-15 ES18_ES':
  Calibration date: 2023-07-20
  Frequency: [18000.0]
  Gain correction: [22.99]
  Sa correction: [-0.0262]
  Equivalent beam angle: -17.0
  Sound speed: 1526.5
  Absorption: 0.002661


In [ ]:
# =============================================================================
# EXAMPLE USAGE: Loading from saved files (for use in a separate session)
# =============================================================================

print("LOADING CALIBRATION FROM SAVED FILES")
print("=" * 80)
print("""
In a new session, you can load the mapping files and retrieve calibration:

    from aa_si_calibration.mapping_algorithm import (
        load_raw_configs, 
        get_calibration_from_file
    )
    import yaml
    
    # Load the mapping dictionary
    with open('mapping_files/channel_to_calibration_mapping.yaml', 'r') as f:
        mapping_dict = yaml.safe_load(f)
    
    # Get calibration from individual files
    cal_data = get_calibration_from_file(
        filename='D20160725-T205832.raw',
        channel_id='GPT  38 kHz 0090720346bc 1-1 ES38B',
        mapping_dict=mapping_dict,
        cal_files_dir='single_channel_calibration_files/'
    )
""")

# Demonstrate loading from individual files
if mapping_dict:
    example_filename = list(mapping_dict.keys())[0]
    if mapping_dict[example_filename]:
        example_channel = list(mapping_dict[example_filename].keys())[0]
        
        # Load from single-channel calibration files
        cal_from_file = get_calibration_from_file(
            example_filename, 
            example_channel, 
            mapping_dict, 
            SINGLE_CAL_OUTPUT
        )
        
        if cal_from_file:
            print(f"\n Successfully loaded calibration from file!")
            print(f"  File: {example_filename}")
            print(f"  Channel: {example_channel}")
            print(f"  Gain: {cal_from_file.get('gain_correction')}")
        else:
            print(f" Could not load calibration from file")

LOADING CALIBRATION FROM SAVED FILES

In a new session, you can load the mapping files and retrieve calibration:

    from aa_si_calibration.mapping_algorithm import (
        load_raw_configs, 
        get_calibration_from_file
    )
    import yaml

    # Load the mapping dictionary
    with open('mapping_files/channel_to_calibration_mapping.yaml', 'r') as f:
        mapping_dict = yaml.safe_load(f)

    # Get calibration from individual files
    cal_data = get_calibration_from_file(
        filename='D20160725-T205832.raw',
        channel_id='GPT  38 kHz 0090720346bc 1-1 ES38B',
        mapping_dict=mapping_dict,
        cal_files_dir='single_channel_calibration_files/'
    )


 Successfully loaded calibration from file!
  File: D20230721-T174615.raw
  Channel: WBT 400479-15 ES18_ES
  Gain: [22.99]


In [ ]:
# =============================================================================
# PIPELINE SUMMARY
# =============================================================================

print("=" * 80)
print("PIPELINE COMPLETE - OUTPUT FILES SUMMARY")
print("=" * 80)

import os

def list_files_recursive(folder, indent=0):
    """List all files in a folder recursively."""
    if not folder.exists():
        return
    for item in sorted(folder.iterdir()):
        if item.is_dir():
            print("  " * indent + f"📁 {item.name}/")
            list_files_recursive(item, indent + 1)
        else:
            size_kb = item.stat().st_size / 1024
            print("  " * indent + f"📄 {item.name} ({size_kb:.1f} KB)")

print(f"\nOutput directory: {OUTPUT_BASE}")
print("-" * 40)
list_files_recursive(OUTPUT_BASE)

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("""
With these output files, you can:

1. Use the raw_file_configs.yaml to understand your data structure
2. Use the single-channel calibration files with any compatible processing tool
3. Use the mapping files to automatically apply the correct calibration
   to each channel when processing raw data with echopype or similar tools
4. Provide your own single-channel calibration files and run the
   user_provided_cal_pipeline.ipynb to map them to raw files

For echopype calibration, you can extract the relevant parameters:
    - gain_correction
    - sa_correction  
    - equivalent_beam_angle
    - sound_speed_indicative
    - absorption_indicative
""")

PIPELINE COMPLETE - OUTPUT FILES SUMMARY

Output directory: Full_Pipeline_Outputs
----------------------------------------
📁 logs/
  📄 calibration_flags.json (0.9 KB)
📁 mapping_files/
  📄 calibration_configurations.yaml (22.5 KB)
  📄 channel_to_calibration_mapping.yaml (0.5 KB)
📁 raw_file_configs/
  📄 raw_file_configs.yaml (6.1 KB)
📁 single_channel_calibration_files/
  📄 2023-07-20__18000__config-1.yml (1.6 KB)
  📄 2023-07-20__200000__config-2.yml (5.4 KB)
  📄 2023-07-20__38000__config-2.yml (8.1 KB)
  📄 2023-07-20__70000__config-2__2.yml (6.4 KB)
📁 unused_calibration_files/
  📄 2023-07-20__200000__config-1.yml (1.7 KB)
  📄 2023-07-20__38000__config-1.yml (1.7 KB)
  📄 2023-07-20__70000__config-1.yml (1.6 KB)

NEXT STEPS

With these output files, you can:

1. Use the raw_file_configs.yaml to understand your data structure
2. Use the single-channel calibration files with any compatible processing tool
3. Use the mapping files to automatically apply the correct calibration
   to each chan